In [ ]:
import richfile as rfs
import numpy as np
import pandas as pd
from scipy import sparse
# register numpy, pandas and scipy types to richfile
if not ("pandas_dataframe" in [prop["type_name"] for prop in rfs.functions._TYPE_LOOKUP]):
    rfs.functions.register_type(
        type_name="pandas_dataframe",
        function_load=lambda path: pd.read_csv(path),
        function_save=lambda path, obj: obj.to_csv(path),
        object_class=pd.DataFrame,
        library="pandas",
        suffix="csv",
    )
if not ("pandas_series" in [prop["type_name"] for prop in rfs.functions._TYPE_LOOKUP]):    
    rfs.functions.register_type(
        type_name="pandas_series",
        function_load=lambda path: pd.read_csv(path, squeeze=True),
        function_save=lambda path, obj: obj.to_csv(path),
        object_class=pd.Series,
        library="pandas",
        suffix="csv",
)
if not ("scipy_sparse" in [prop["type_name"] for prop in rfs.functions._TYPE_LOOKUP]):
    rfs.functions.register_type(
        type_name="scipy_sparse",
        function_load=lambda path: sparse.load_npz(path),
        function_save=lambda path, obj: sparse.save_npz(path, obj),
        object_class=sparse.csr_matrix,
        library="scipy",
        suffix="npz",
)
if not ("numpy_array" in [prop["type_name"] for prop in rfs.functions._TYPE_LOOKUP]):
    rfs.functions.register_type(
        type_name="numpy_array",
        function_load=lambda path: np.load(path, allow_pickle=True),
        function_save=lambda path, obj: np.save(path, obj),
        object_class=np.ndarray,
        library="numpy",
        suffix="npy",
    )
# add scipy_sparse_array
if not ("scipy_sparse_array" in [prop["type_name"] for prop in rfs.functions._TYPE_LOOKUP]):
    rfs.functions.register_type(
        type_name="scipy_sparse_array",
        function_load=lambda path: sparse.load_npz(path),
        function_save=lambda path, obj: sparse.save_npz(path, obj),
        object_class=sparse.spmatrix,
        library="scipy",
        suffix="npz",
    )


In [ ]:
data_path = r'C:\Data\datasets\bmi_packets\bmi_packets'
r = rfs.RichFile(rf"{data_path}\mouse_0403L.richfile")  ## Path to an existing richfile
r.view_directory_tree()

In [ ]:
# let print all the high level folders and files in the richfile
for item in r.keys():
    print(item)

In [ ]:
# inspect neural data
neural_data = r["data_neural_aligned"]
print("Neural data subfolders and files:")
for itnm in neural_data.keys():
    print(itnm)

# inspect behavior data one day of data
one_day_neural_data = neural_data["20230702"].load()
print(f"one day neural data type: {type(one_day_neural_data)}")
print(f"one day neural data shape: {one_day_neural_data.shape}")

# let plot the first 1000 samples of the firast 10 channels of the neural data
import matplotlib.pyplot as plt
for i in range(10):
    plt.plot(one_day_neural_data[:1000, i]+i*90, label=f"Channel {i}")
plt.title("First 1000 samples of the first 10 channels of neural data")
plt.xlabel("Samples (30 Hz)")
plt.ylabel("Firing rate (Hz)")
plt.show()

# let just plot one day
plt.plot(one_day_neural_data[:100, 1])
plt.title("First 1000 samples of the first channel of neural data")
plt.xlabel("Samples (30 Hz)")
plt.ylabel("Firing rate (Hz)")

In [ ]:
weights_day0_idxFactors = r["weights_day0_idxFactors"].load()
print(f"weights_day0_idxFactors type: {type(weights_day0_idxFactors)}")
print(f"weights_day0_idxFactors shape: {weights_day0_idxFactors.shape}")
waights_day0_idxFactors_df = pd.DataFrame(weights_day0_idxFactors)
waights_day0_idxFactors_df
# this is the weights of each factor for each neuron. The NaN values are for the neurons present in day zero.

In [ ]:
# let inspect the lick_times
lick_times = r["lick_times"]
print("lick_times subfolders and files:")
for item in lick_times.keys():
    print(item)

lick_times_one_day = lick_times["20230702"].load()
print(f"lick_times_one_day type: {type(lick_times_one_day)}")
print(f"lick_times_one_day shape: {lick_times_one_day.shape}")
lick_times_series = pd.Series(lick_times_one_day)
lick_times_series 
# time of the licks in samples (30 Hz) for each day.


In [ ]:
#let inspect reward_times_typed
reward_times_typed = r["reward_times_typed"].load()
print(f"reward_times_typed type: {type(reward_times_typed)}")
print(f"reward_times_typed keys: {reward_times_typed.keys()}")
reward_times_typed_one_day = reward_times_typed["20230707"]
print(f"reward_times_typed_one_day type: {type(reward_times_typed_one_day)}")
print(f"reward_times_typed_one_day keys: {reward_times_typed_one_day.keys()}")
reward_times_typed_one_day[(1, 1, 1, 1)]

# this is the time of the rewards in samples (30 Hz) for each day. The keys are tuples of 
# ( whether cursor on, whther the feedback on, whether the reward was delivered\ whether reward on, whether the reward was collected\ whther licked) 


In [ ]:
data_ws = r["data_ws"].load()
print(f"data_ws type: {type(data_ws)}")
print(f"data_ws keys: {data_ws.keys()}")
data_ws_one_day = data_ws["20230725"]
print(f"data_ws_one_day type: {type(data_ws_one_day)}")
print(f"data_ws_one_day keys: {data_ws_one_day.keys()}")
rewards = data_ws_one_day['rewards']
print(f"cursor type: {type(rewards)}")
print(f"cursor shape: {rewards.shape}")
rewards_series = pd.Series(rewards)
rewards_series

# these are bunch of signals from the wavesurfer:
#'sync_pulse', 'laser_pickoff' (this is the laser pickoff signal power, the ideal is that to be stabel),
# 'treadmill', 'lickometer', 'rewards', 'cursor', 'y_galvo'


In [ ]:
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
cursor = data_ws_one_day["cursor"]
rewards = data_ws_one_day["rewards"]
time_set = np.arange(len(cursor)) / 30.0  # time in seconds, assuming 30 Hz sampling rate

# Mild smoothing
# sigma is in samples. At 30 Hz, sigma=10 is ~670 ms.
cursor_smooth = gaussian_filter1d(cursor.astype(float), sigma=10)

threshold = 0.5 * np.nanmax(rewards)
rewards_binary = rewards > threshold

k = 1 * 60 * 30
start_time = 10 * 60 * 30

plt.figure(figsize=(12, 4))
plt.plot(time_set[start_time:start_time+k], cursor_smooth[start_time:start_time+k], label="cursor")
plt.plot(time_set[start_time:start_time+k], cursor[start_time:start_time+k], label="cursor raw", alpha=0.3, color="gray")
plt.step(
    time_set[start_time:start_time+k],
    rewards_binary[start_time:start_time+k].astype(float)*np.max(cursor_smooth[start_time:start_time+k]),
    where="post",
    label="reward binary",
)
plt.title("Cursor and rewards signals from WaveSurfer (smoothed)")
plt.legend()
plt.show()


In [ ]:
date_day0 = r["date_day0"].load()
print(f"date_day0 type: {type(date_day0)}")
date_day0
# when was the last day of the baseline recordding (it's usualy more than one day of baseline)

In [ ]:
extras = r["extras"]
extras
# these are shit towm of etera prams, logger and other meta stuff you may need

In [ ]:
#extras['expParams']['20230709'][extra]

In [ ]:
#[print(x.load()) for x in extras['expParams']['20230709']['loggerNames']['trials']]

In [ ]:
xx = r["extras"]["expParams"]['20230725']['baselineStuff']['factor_to_use'].load()
xx

In [ ]:
xx['directory']
#np.shape(xx['factor_space'])